This simple RAG system provides a solid foundation for building more complex information retrieval and question-answering systems. By encoding document content into a searchable vector store, it enables efficient retrieval of relevant information in response to queries. This approach is particularly useful for applications requiring quick access to specific information within large documents or document collections.

***Package Installation and Imports***

The cell below installs all necessary packages required to run this notebook.

In [1]:
# %pip install rank_bm25==0.2.2 PyMuPDF==1.26.1 faiss-cpu==1.11.0 python-dotenv==1.1.0 pypdf==5.6.0 langchain-openai==0.3.23 deepeval==3.1.0

In [2]:
# Clone the repository from github 
# !git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
# sys.path.append('RAG_TECHNIQUES')

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser


In [4]:
import deepeval
print(deepeval.__version__)

3.1.0


In [5]:
from typing import List, Tuple, Dict, Any

from deepeval import evaluate
from deepeval.metrics import GEval, FaithfulnessMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

In [6]:
import os
import sys
from dotenv import load_dotenv
#from google.colab import userdata

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Set environment variables (especially useful for LangChain integrations)
openai_api_key = os.environ["OPENAI_API_KEY"]

print("API Keys loaded (partially hidden for security):")
print(f"OpenAI Key starts with: {openai_api_key[:10]}...")

API Keys loaded (partially hidden for security):
OpenAI Key starts with: sk-proj-9u...


In [7]:
from evaluation.evalute_rag import evaluate_rag

from helper_functions import (EmbeddingProvider,
                              retrieve_context_per_question,
                              replace_t_with_space,
                              get_langchain_embedding_provider,
                              show_context)

# Read Docs

In [8]:
import os
import urllib.request

os.makedirs('data', exist_ok=True)

url = "https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf"
urllib.request.urlretrieve(url, "data/Understanding_Climate_Change.pdf")
print("Downloaded")

Downloaded


In [9]:
path = "data/Understanding_Climate_Change.pdf"

# Encode document

In [10]:
def encode_pdf(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A FAISS vector store containing the encoded book content.
    """

    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings (Tested with OpenAI and Amazon Bedrock)
    embeddings = get_langchain_embedding_provider(EmbeddingProvider.OPENAI)
    #embeddings = get_langchain_embedding_provider(EmbeddingProvider.AMAZON_BEDROCK)

    # Create vector store
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore



In [11]:
chunks_vector_store = encode_pdf(path, chunk_size=1000, chunk_overlap=200)

# Create retriever

In [12]:
chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 2})

# Test retriever

In [13]:
test_query = "What is the main cause of climate change?"
context = retrieve_context_per_question(test_query, chunks_query_retriever)
show_context(context)

D:\ML DL Projects\RAG-Techs\helper_functions.py:143: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = chunks_query_retriever.get_relevant_documents(question)


Context 1:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal


Context 2:
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch,

# Evaluate results

In [14]:
#Note - this currently works with OPENAI only
evaluate_rag(chunks_query_retriever)

{'questions': ['1. **Multiple Choice Question:**',
  '   What is the primary cause of the recent acceleration in climate change?',
  '   A) Solar radiation variations',
  '   B) Natural geological processes',
  '   C) Human-induced emissions of greenhouse gases',
  '   D) Increased volcanic activity',
  '',
  '2. **True/False Question:**',
  '   True or False: The melting of polar ice caps has no effect on sea level rise.',
  '',
  '3. **Short Answer Question:**',
  '   Explain how deforestation contributes to climate change.',
  '',
  '4. **Data Interpretation Question:**',
  '   Given a graph showing the levels of carbon dioxide in the atmosphere over the last 100 years, identify the trend and discuss two potential impacts of this trend on global ecosystems.',
  '',
  '5. **Essay Question:**',
  '   Discuss the role of international cooperation in addressing climate change. Include examples of at least two international agreements aimed at reducing carbon emissions and assess their e